# 01 — Data Acquisition & Exploration

**Phase 1 deliverable.**

Documents the structure and quirks of CVM DADOS_ABERTOS CSVs for the 49 largest B3 companies, 2022–2025.

**Data sources:**
- `https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/ITR/DADOS/` — quarterly (ITR) ZIPs
- `https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/` — annual (DFP) ZIPs
- `https://dados.cvm.gov.br/dados/CIA_ABERTA/CAD/DADOS/` — company registration cadaster

In [1]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

ROOT = Path("..")  # notebook runs from notebooks/
RAW = ROOT / "data" / "raw"
CSVS = RAW / "csvs"

## 1. Manifest overview

In [2]:
manifest = json.loads((RAW / "manifest.json").read_text())
print(f"Total filings:    {len(manifest)}")
print(f"Unique companies: {len({m['ticker'] for m in manifest})}")

by_type = Counter((m['filing_type'], m['reference_date'][:4]) for m in manifest)
df_type = pd.DataFrame(
    [(t, y, c) for (t,y), c in sorted(by_type.items())],
    columns=['type', 'year', 'count']
).pivot(index='year', columns='type', values='count').fillna(0).astype(int)
print("\nFilings per year & type:")
display(df_type)

Total filings:    686
Unique companies: 49

Filings per year & type:


type,DFP,ITR
year,,
2022,61,0
2023,58,179
2024,63,166
2025,0,159


## 2. CSV schema

In [3]:
# Financial statement CSV columns
dre = pd.read_csv(CSVS / "itr_2024" / "itr_cia_aberta_DRE_con_2024.csv", sep=";", encoding="utf-8")
print("DRE columns:", list(dre.columns))
print(f"Shape: {dre.shape}")
print("\nORDEM_EXERC values:", dre['ORDEM_EXERC'].unique())
print("ESCALA_MOEDA values:", dre['ESCALA_MOEDA'].unique())
dre.head(3)

DRE columns: ['CNPJ_CIA', 'DT_REFER', 'VERSAO', 'DENOM_CIA', 'CD_CVM', 'GRUPO_DFP', 'MOEDA', 'ESCALA_MOEDA', 'ORDEM_EXERC', 'DT_INI_EXERC', 'DT_FIM_EXERC', 'CD_CONTA', 'DS_CONTA', 'VL_CONTA', 'ST_CONTA_FIXA']
Shape: (17432, 15)

ORDEM_EXERC values: <ArrowStringArray>
['PENÚLTIMO', 'ÚLTIMO']
Length: 2, dtype: str
ESCALA_MOEDA values: <ArrowStringArray>
['MIL']
Length: 1, dtype: str


,CNPJ_CIA,DT_REFER,VERSAO,DENOM_CIA,CD_CVM,GRUPO_DFP,MOEDA,ESCALA_MOEDA,ORDEM_EXERC,DT_INI_EXERC,DT_FIM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA,ST_CONTA_FIXA
0,00.000.000/0001-91,2024-03-31,1,BCO BRASIL S.A.,1023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2023-01-01,2023-03-31,3.01,Receitas de Intermediação Financeira,65909746.0,S
1,00.000.000/0001-91,2024-03-31,1,BCO BRASIL S.A.,1023,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2024-01-01,2024-03-31,3.01,Receitas de Intermediação Financeira,66655193.0,S
2,00.000.000/0001-91,2024-03-31,1,BCO BRASIL S.A.,1023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2023-01-01,2023-03-31,3.01.01,Receita de Juros,65909746.0,N


## 3. Target companies

In [4]:
resolved = json.loads((RAW / "resolved_companies.json").read_text())
df_co = pd.DataFrame(resolved)
print(f"Resolved: {len(df_co)}/50 target companies")
display(df_co[['ticker','name','sector','cnpj','cd_cvm']].sort_values('ticker'))

Resolved: 49/50 target companies


,ticker,name,sector,cnpj,cd_cvm
4,ABEV3,AMBEV S.A.,Bebidas,07.526.557/0001-00,23264
43,ALPA4,ALPARGATAS SA,Calçados,61.079.117/0001-05,10456
5,B3SA3,"B3 S.A. - BRASIL, BOLSA, BALCÃO",Serviços Financeiros,09.346.601/0001-25,21610
8,BBAS3,BANCO DO BRASIL S.A.,Financeiro,00.000.000/0001-91,1023
3,BBDC4,BANCO BRADESCO S.A.,Financeiro,60.746.948/0001-12,906
36,BEEF3,MINERVA S/A,Alimentos,67.620.377/0001-14,20931
46,BRAP4,BRADESPAR S/A,Holdings,03.847.461/0001-92,18724
17,BRFS3,BRF S.A.,Alimentos,01.838.723/0001-27,16292
23,CMIG4,CIA ENERG MINAS GERAIS - CEMIG,Energia Elétrica,17.155.730/0001-64,2453
33,CMIN3,CSN MINERAÇÃO S.A.,Mineração,08.902.291/0001-15,25585


## 4. Sample financial values — Petrobras

In [5]:
PETR_CNPJ = "33.000.167/0001-01"

dre_petr = dre[
    (dre['CNPJ_CIA'] == PETR_CNPJ) &
    (dre['ORDEM_EXERC'] == 'ÚLTIMO')
][['DT_REFER', 'CD_CONTA', 'DS_CONTA', 'VL_CONTA']].copy()

# Values are in R$ MIL (thousands)
dre_petr['VL_CONTA_BRL'] = dre_petr['VL_CONTA'].astype(float) * 1000

key_accounts = ['3.01', '3.02', '3.03', '3.05', '3.11']
display(dre_petr[dre_petr['CD_CONTA'].isin(key_accounts)].sort_values('CD_CONTA'))

,DT_REFER,CD_CONTA,DS_CONTA,VL_CONTA,VL_CONTA_BRL
8429,2024-03-31,3.01,Receita de Venda de Bens e/ou Serviços,117721000.0,1.177210e+11
8516,2024-06-30,3.01,Receita de Venda de Bens e/ou Serviços,239979000.0,2.399790e+11
8517,2024-06-30,3.01,Receita de Venda de Bens e/ou Serviços,122258000.0,1.222580e+11
8688,2024-09-30,3.01,Receita de Venda de Bens e/ou Serviços,369561000.0,3.695610e+11
8689,2024-09-30,3.01,Receita de Venda de Bens e/ou Serviços,129582000.0,1.295820e+11
8431,2024-03-31,3.02,Custo dos Bens e/ou Serviços Vendidos,-57020000.0,-5.702000e+10
8520,2024-06-30,3.02,Custo dos Bens e/ou Serviços Vendidos,-118231000.0,-1.182310e+11
8521,2024-06-30,3.02,Custo dos Bens e/ou Serviços Vendidos,-61211000.0,-6.121100e+10
8692,2024-09-30,3.02,Custo dos Bens e/ou Serviços Vendidos,-181235000.0,-1.812350e+11
8693,2024-09-30,3.02,Custo dos Bens e/ou Serviços Vendidos,-63004000.0,-6.300400e+10


## 5. Data quirks

Key observations from exploring the raw data:

1. **Separator**: `;` (not comma). Encoding: `latin-1` in raw ZIPs, `utf-8` after our filtering.
2. **Scale**: All values in `ESCALA_MOEDA = MIL` (thousands of BRL). Multiply by 1,000 for absolute values.
3. **Current vs prior period**: `ORDEM_EXERC = 'ÚLTIMO'` is the current filing period. `'PENÚLTIMO'` is the prior comparison period (same data appears twice). Always filter to `ÚLTIMO`.
4. **Account codes**: Standard CVM chart of accounts — `3.01` = Receita Líquida, `3.02` = COGS, `3.03` = Gross Profit, `3.11` = Net Income.
5. **Versioning**: Multiple `VERSAO` values possible for the same `DT_REFER`. Use the highest version.
6. **PDF access**: The `LINK_DOC` URLs point to `rad.cvm.gov.br` which blocks automated access from VPN/cloud IPs. PDFs must be downloaded from a residential/office network connection.
7. **EBITDA and Net Debt**: Not standardised CVM account codes — must be computed or extracted from notes (Notas Explicativas) in the PDF.
8. **CCR (CCRO3)**: Not present in the CVM DADOS_ABERTOS cadaster — likely filed under a holding structure. Excluded from the 49-company set.